<a href="https://colab.research.google.com/github/Shikher-jain/Hugging_Face_LLM/blob/main/Transformer/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==4.41.2 accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
import torch

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device}")

In [ ]:
# HF_TOKEN = userdata.get("HF_TOKEN")
# login(HF_TOKEN)

In [ ]:

import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter token: ")

In [ ]:
# -------------------------
# Sentiment
# -------------------------
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

# -------------------------
# Zero-shot
# -------------------------
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

# -------------------------
# Text Generation
# -------------------------
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M",
    device=device
)

# -------------------------
# Fill-mask
# -------------------------
unmasker = pipeline("fill-mask", model="distilbert/distilroberta-base", device=device)

# -------------------------
# NER
# -------------------------
# ner = pipeline("ner", grouped_entities=True)
# ner = pipeline("ner",aggregation_strategy="simple")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)


# -------------------------
# QA
# -------------------------
qa = pipeline("question-answering")

# -------------------------
# Summarization
# -------------------------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# -------------------------
# Translation
# -------------------------
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")

In [ ]:
print(sentiment_classifier("I've been waiting for a HuggingFace course my whole life."))
print(sentiment_classifier([
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!"
]))

In [ ]:
text = "The new AI model outperforms previous benchmarks"
labels = ["sports", "technology", "politics"]

print(zero_shot_classifier(text, candidate_labels=labels))

print(zero_shot_classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"]
))

In [ ]:
generator("In this course, we will teach you how to")

In [ ]:
unmasker("This course will teach you all about <mask> models.", top_k=2)

In [ ]:
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
    num_beams=2,
    do_sample=True,
    pad_token_id=generator.model.config.eos_token_id
)


In [ ]:
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

In [ ]:
qa(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

In [ ]:
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)

In [ ]:
translator("Ce cours est produit par Hugging Face.")

In [ ]:
image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224",device=device
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

In [ ]:
transcriber = pipeline(
task="automatic-speech-recognition", model="openai/whisper-large-v3",device=device)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)
